# HSE seminar 3, task 1: decision tree by entropy

Source: `../Machine-Learning/Семинар 3. Решающие деревья.pdf`, Задание №1. PRML link: Ex. 1.39, 1.41 (information gain = mutual information), §1.5 (threshold 0.4).

Depth-2 tree predicting Успех (success) from Реклама (ads), Опыт (experience), Здоровье (health); split criterion: entropy; leaf predicts success if the share of successes > 0.4.

1. Own `entropy` / `info_gain` in numpy, checked on the Ex. 1.39 table (0.637, 0.462, 0.174 nats).
2. Build the tree by hand; draw it with the success share in each leaf.
3. Predict: experienced team, no ads, health topic.
4. Check against sklearn `DecisionTreeClassifier(criterion="entropy", max_depth=2)`: `tree_.feature`, `tree_.impurity` (bits, not nats).

In [1]:
import pandas as pd

df = pd.DataFrame({
    "success":    [1, 0, 1, 0, 1, 0, 1, 0],
    "ads":        [1, 1, 0, 0, 1, 0, 0, 0],
    "experience": [1, 1, 1, 0, 1, 0, 1, 1],
    "health":     [0, 1, 1, 1, 0, 0, 0, 0],
})

In [8]:
import numpy as np

class Model:
    X: pd.DataFrame
    Y: pd.Series


    def fit(self, X, Y):
        self.X = X
        self.Y = Y
        self.split = self.best_split(X, Y, X.keys())


    def predict_proba(self, x: pd.Series):
        fk = self.split[0]
        fv = x[fk]
        ffk = self.split[1][fv]
        ffv = x[ffk]

        # prob

        X, Y = self.X, self.Y
        mask = X[fk] == fv
        X, Y = X[mask], Y[mask]
        Y_branch = Y                                   # new: the branch's rows, kept for the fallback
        mask = X[ffk] == ffv
        X, Y = X[mask], Y[mask]

        if Y.size == 0:                                # new: empty leaf, no training rows reach it
            return (Y_branch == 1).sum() / Y_branch.size   # new: fall back to the parent branch's share
        # count X==fffv AND Y===1
        f_one = (Y == 1).sum()
        return f_one / Y.size

    def predict(self, x):
        return self.predict_proba(x) > 0.4

    def best_split(self, X, Y, features):
        # gain = info_gain(X, features) # return array or iterat? i think better to use vector functions as much as possible
        entropies = self.info_gain(X, Y, features)

        max_entropy_feature = max(entropies, key=lambda k: entropies[k])

        splits = (max_entropy_feature, [None, None])

        features = [f for f in features if f != max_entropy_feature]

        feature_vals = X[max_entropy_feature].unique()

        for f in feature_vals:

            mask = X[max_entropy_feature] == f

            entropies = self.info_gain(X[mask], Y[mask], features)

            ss = splits[1]
            ss[f] = max(entropies, key=lambda k: entropies[k])
            splits = (splits[0], ss)

        return splits

    def info_gain(self, X, Y, features):
        Y_entropy = self.entropy(Y)
        X_entropy = {}
        for f in features:
            X_entropy[f] = Y_entropy

            feature_vals = X[f].unique()

            for v in feature_vals:
                mask = X[f] == v
                X_entropy[f] = X_entropy[f] - Y[mask].size * self.entropy(Y[mask]) / Y.size

        return X_entropy

    def entropy(self, Y):
        mu = Y.sum() / Y.size

        def p(y):
            return mu**y*(1-mu)**(1-y)

        if mu == 0:
            # prob of 0 is 100%
            return -p(0)*np.log(p(0))

        if mu == 1:
            # prob of 1 is 100%
            return -p(1)*np.log(p(1))

        return -p(0)*np.log(p(0))-p(1)*np.log(p(1))






In [3]:
m = Model()

cases = [
    (pd.Series([0, 1]),          np.log(2)),                                    # fair coin: max for binary
    (pd.Series([1, 1, 1]),       0.0),                                          # pure, all 1
    (pd.Series([0, 0, 0, 0]),    0.0),                                          # pure, all 0
    (pd.Series([0, 0, 1]),       -(2/3)*np.log(2/3) - (1/3)*np.log(1/3)),       # Ex. 1.39 H[x] = 0.637
    (df["success"],              np.log(2)),                                    # 4 of 8
    (pd.Series([1, 0, 0, 0, 0]), -(1/5)*np.log(1/5) - (4/5)*np.log(4/5)),       # unbalanced: 0.500
]

for y, expected in cases:
    got = m.entropy(y)
    assert np.isclose(got, expected), f"{list(y)}: got {got}, expected {expected}"
    print(f"{list(y)!s:28} {got:.3f}  ok")

[0, 1]                       0.693  ok
[1, 1, 1]                    -0.000  ok
[0, 0, 0, 0]                 -0.000  ok
[0, 0, 1]                    0.637  ok
[1, 0, 1, 0, 1, 0, 1, 0]     0.693  ok
[1, 0, 0, 0, 0]              0.500  ok


In [4]:
m = Model()


def H(p):  # entropy of a Bernoulli(p) in nats, written out by hand
    return -p*np.log(p) - (1-p)*np.log(1-p)

F = ["ads", "experience", "health"]
small = pd.DataFrame({"y": [0, 1, 0, 1], "same": [0, 1, 0, 1], "indep": [0, 0, 1, 1], "const": [1, 1, 1, 1]})
ex139 = pd.DataFrame({"x": [0, 0, 1], "y": [0, 1, 1]})   # Ex. 1.39 table: three cells of 1/3 each

ig_cases = [
    # (name, X, Y, feature, expected)
    ("ads, all 8 rows",        df[F], df["success"], "ads",        np.log(2) - (3/8)*H(1/3) - (5/8)*H(2/5)),
    ("experience, all 8 rows", df[F], df["success"], "experience", np.log(2) - (6/8)*H(2/3)),   # e = 0 branch is pure
    ("health, all 8 rows",     df[F], df["success"], "health",     np.log(2) - (3/8)*H(1/3) - (5/8)*H(3/5)),
    ("experience inside ads=1", df[F][df["ads"] == 1], df["success"][df["ads"] == 1], "experience", 0.0),  # constant there
    ("feature = target",       small, small["y"], "same",  np.log(2)),   # knowing x removes all uncertainty: I = H[y]
    ("independent feature",    small, small["y"], "indep", 0.0),         # each branch is still 50/50
    ("constant feature",       small, small["y"], "const", 0.0),         # one branch, nothing learned
    ("Ex. 1.39 table",         ex139, ex139["y"], "x", np.log(3) - (4/3)*np.log(2)),   # I[x,y] = 0.174
]

for name, X, Y, f, expected in ig_cases:
    got = m.info_gain(X, Y, [f])[f]
    assert np.isclose(got, expected), f"{name}: got {got}, expected {expected}"
    print(f"{name:26} {got:.3f}  ok")

ads, all 8 rows            0.034  ok
experience, all 8 rows     0.216  ok
health, all 8 rows         0.034  ok
experience inside ads=1    0.000  ok
feature = target           0.693  ok
independent feature        0.000  ok
constant feature           0.000  ok
Ex. 1.39 table             0.174  ok


In [5]:
m = Model()

# Hand-built case where the two branches must choose different features.
#   root b: a and c both have gain 0 on all 6 rows (every branch stays 50/50), b has gain > 0
#   b = 0 branch (rows 0, 2, 5): c is constant there -> gain 0, so a wins
#   b = 1 branch (rows 1, 3, 4): a is constant there -> gain 0, so c wins
toy = pd.DataFrame({
    "a": [0, 1, 1, 1, 1, 0],
    "b": [0, 1, 0, 1, 1, 0],
    "c": [0, 1, 0, 0, 1, 0],
    "y": [1, 1, 1, 0, 0, 0],
})
root, branches = m.best_split(toy[["a", "b", "c"]], toy["y"], ["a", "b", "c"])
assert root == "b", f"toy root: got {root}"
assert branches[0] == "a", f"toy branch b=0: got {branches[0]}"
assert branches[1] == "c", f"toy branch b=1: got {branches[1]}"
print("toy: different feature per branch  ok")

# Seminar data. Root: experience has the largest gain (see the info_gain tests).
#   experience = 1 branch (6 rows): ads gain 0, health gain > 0, so health wins
#   experience = 0 branch (2 rows): both failures, pure, every gain is 0; any choice is fine
F = ["ads", "experience", "health"]
root, branches = m.best_split(df[F], df["success"], F)
assert root == "experience", f"seminar root: got {root}"
assert branches[1] == "health", f"seminar branch experience=1: got {branches[1]}"
assert branches[0] in ("ads", "health"), f"seminar branch experience=0: got {branches[0]}"
print("seminar tree                       ok")

toy: different feature per branch  ok
seminar tree                       ok


In [6]:
m = Model()
F = ["ads", "experience", "health"]
m.fit(df[F], df["success"])

# Leaf shares counted by hand from the 8 rows (root experience; experience = 1 branch splits on health):
#   experience = 1, health = 1: rows 1, 2       -> success 0, 1       -> 1/2
#   experience = 1, health = 0: rows 0, 4, 6, 7 -> success 1, 1, 1, 0 -> 3/4
#   experience = 0: rows 3, 5, both failures    -> 0 (whichever feature that branch split on)
pp_cases = [
    # (x, expected p(success), expected decision > 0.4)
    ({"experience": 1, "ads": 0, "health": 1}, 1/2, True),    # the seminar question
    ({"experience": 1, "ads": 1, "health": 1}, 1/2, True),    # ads differs: same leaf, same answer
    ({"experience": 1, "ads": 0, "health": 0}, 3/4, True),
    ({"experience": 1, "ads": 1, "health": 0}, 3/4, True),
    ({"experience": 0, "ads": 0, "health": 0}, 0.0, False),
]

for x, expected, decision in pp_cases:
    x = pd.Series(x)
    got = m.predict_proba(x)
    assert np.isclose(got, expected), f"{dict(x)}: got {got}, expected {expected}"
    assert m.predict(x) == decision, f"{dict(x)}: predict got {m.predict(x)}, expected {decision}"
    print(f"{dict(x)!s:45} {got:.3f}  ok")

{'experience': 1, 'ads': 0, 'health': 1}      0.500  ok
{'experience': 1, 'ads': 1, 'health': 1}      0.500  ok
{'experience': 1, 'ads': 0, 'health': 0}      0.750  ok
{'experience': 1, 'ads': 1, 'health': 0}      0.750  ok
{'experience': 0, 'ads': 0, 'health': 0}      0.000  ok


In [9]:
from itertools import product
from sklearn.tree import DecisionTreeClassifier

F = ["ads", "experience", "health"]
m = Model()
m.fit(df[F], df["success"])
clf = DecisionTreeClassifier(criterion="entropy", max_depth=2, random_state=0).fit(df[F], df["success"])

# Same root feature? tree_.feature[0] is the column index used at the root node.
assert F[clf.tree_.feature[0]] == m.split[0], f"root: sklearn {F[clf.tree_.feature[0]]}, mine {m.split[0]}"

# Same probability and decision for every possible object: all 2^3 combinations.
grid = pd.DataFrame(list(product([0, 1], repeat=3)), columns=F)
sk_proba = clf.predict_proba(grid)[:, 1]          # column 1 = p(success)
for i, row in grid.iterrows():
    mine = m.predict_proba(row)
    assert not np.isnan(mine), f"{dict(row)}: nan, an empty leaf; add a fallback (sklearn gives {sk_proba[i]})"
    assert np.isclose(mine, sk_proba[i]), f"{dict(row)}: mine {mine}, sklearn {sk_proba[i]}"
    assert m.predict(row) == (sk_proba[i] > 0.4), f"{dict(row)}: decision differs"
    print(f"{dict(row)!s:45} {mine:.3f}  ok")

{'ads': 0, 'experience': 0, 'health': 0}      0.000  ok
{'ads': 0, 'experience': 0, 'health': 1}      0.000  ok
{'ads': 0, 'experience': 1, 'health': 0}      0.750  ok
{'ads': 0, 'experience': 1, 'health': 1}      0.500  ok
{'ads': 1, 'experience': 0, 'health': 0}      0.000  ok
{'ads': 1, 'experience': 0, 'health': 1}      0.000  ok
{'ads': 1, 'experience': 1, 'health': 0}      0.750  ok
{'ads': 1, 'experience': 1, 'health': 1}      0.500  ok


In [10]:
m = Model()
F = ["ads", "experience", "health"]
m.fit(df[F], df["success"])

root, branches = m.split
X, Y = m.X, m.Y
print(f"{root}   (n = {Y.size}, share of successes {Y.mean():.2f})")
for v1 in (1, 0):
    mask1 = X[root] == v1
    second = branches[v1]
    print(f"├─ {root} = {v1}: ask {second}   (n = {mask1.sum()}, share {Y[mask1].mean():.2f})")
    for v2 in (1, 0):
        leaf = mask1 & (X[second] == v2)
        n = leaf.sum()
        share = Y[leaf].mean() if n > 0 else Y[mask1].mean()     # empty leaf: parent branch's share
        note = "" if n > 0 else "  (empty leaf, parent's share)"
        decision = "success" if share > 0.4 else "failure"
        print(f"│   ├─ {second} = {v2}: share {share:.2f} -> {decision}   (n = {n}){note}")

experience   (n = 8, share of successes 0.50)
├─ experience = 1: ask health   (n = 6, share 0.67)
│   ├─ health = 1: share 0.50 -> success   (n = 2)
│   ├─ health = 0: share 0.75 -> success   (n = 4)
├─ experience = 0: ask ads   (n = 2, share 0.00)
│   ├─ ads = 1: share 0.00 -> failure   (n = 0)  (empty leaf, parent's share)
│   ├─ ads = 0: share 0.00 -> failure   (n = 2)


## Answers

**1. Tree** (depth 2, entropy; share of successes in each leaf):

```
experience                      (8 rows, 0.50)
├─ experience = 1 → health      (6 rows, 0.67)
│   ├─ health = 1 → 0.50        (2 rows)
│   └─ health = 0 → 0.75        (4 rows)
└─ experience = 0 → 0.00        (2 rows, pure: no further split needed)
```

My code also splits the pure experience = 0 branch (on ads, a tie at gain 0), with an empty leaf for ads = 1 that falls back to the branch share 0.00. Predictions are the same as sklearn's, which stops at the pure node.

**2. Prediction** for an experienced team, no ads, health topic: experience = 1 → health = 1 → share 0.50 > 0.4, so the startup is predicted to **succeed**. Ads is never asked on this path.